**IMPROVEMENTS**

* [] pass the json for validation directly to the model invocation when possible (check if it's supported) i.e. stop to do the validation after the invocation

Self healing strategies:

* [x] increase temperature error after error (already implemented, works sometimes for qwen3)
* [ ] pass the error message when problems in the json are detected to see if it fixes them afterwards (new prompt with schema, invalid JSON, validation error and "Fix the JSON so it passes validation. Do not change values unless required to fix an error.")

In [1]:
import sys
from groq import Groq
import json
import pydantic 
from pydantic import BaseModel
from typing import get_args, Literal
import os
from google import genai
from openai import OpenAI
from time import sleep

from prompts import ORIGINAL_SYSTEM_PROMPT
from dataset_utils import load_or_download_dataset
from models import build_crf_model
from crf_utils import build_compact_schema, save_prediction, create_base_messages, invoke_model
from api_keys import *

In [2]:
LANGUAGE = "it"
# MODEL_ID = "qwen/qwen3-32b"
# MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
# MODEL_ID = "openai/gpt-oss-120b"
# MODEL_ID = "gemma-3-27b-it"
MODEL_ID = "gemini-3-flash-preview" # "gemini-3-pro-review"
# MODEL_ID = "gpt-5-nano"
if "gemini" in MODEL_ID:
    LLM_TYPE = "gemini"
elif "gemma" in MODEL_ID:
    LLM_TYPE = "gemma"
elif "gpt" in MODEL_ID and "oss" not in MODEL_ID:
    LLM_TYPE = "openai"
else:
    LLM_TYPE = "groq"
OUTPUT_FILE = f"{MODEL_ID.split('/')[-1]}_new-prompt_{LANGUAGE}.jsonl"
OUTPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", OUTPUT_FILE)

FS_DATASET = "NLP-FBK/dyspnea-crf-train"
DATASET_TO_CRF = "NLP-FBK/dyspnea-crf-development"
NUM_FS_EXAMPLES = 3
MAX_RETRIES = 3

MODEL_ID, LLM_TYPE

('gemini-3-flash-preview', 'gemini')

In [3]:
os.chdir("..")

In [4]:
# one-time initializations
if LLM_TYPE=="groq":
    client = Groq(api_key=GROQ_API_KEY)
elif LLM_TYPE in ["gemma", "gemini"]:
    client = genai.Client(api_key=GOOGLE_API_KEY)
elif LLM_TYPE=="openai":
    client = OpenAI(api_key=OPENAI_API_KEY)
CRFModel = build_crf_model()

# Load development dataset
ds = load_or_download_dataset(DATASET_TO_CRF)[LANGUAGE]
print(f"{DATASET_TO_CRF} has {len(ds)} documents")

base_messages, sys_instruction = create_base_messages(CRFModel, ORIGINAL_SYSTEM_PROMPT, FS_DATASET, LANGUAGE, NUM_FS_EXAMPLES, llm_type=LLM_TYPE)
base_messages[-1]

Found local cache at data/dyspnea-valid-options, loading from disk...
Dataset({
    features: ['options', 'item'],
    num_rows: 134
})
Found local cache at data/dyspnea-crf-development, loading from disk...
NLP-FBK/dyspnea-crf-development has 80 documents
Found local cache at data/dyspnea-crf-train, loading from disk...


Content(
  parts=[
    Part(
      text='{"history of recent trauma": "y", "cardiovascular diseases": "y"}'
    ),
  ],
  role='model'
)

In [10]:
# in case multiple runs are required for the same elaboraiton
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["document_id"] for line in f}
len(processed_ids), processed_ids

(43,
 {'1014081_it',
  '1117595_it',
  '114284_it',
  '1147182_it',
  '1166267_it',
  '1231372_it',
  '1236585_it',
  '1258026_it',
  '1307609_it',
  '1323348_it',
  '1323973_it',
  '1350262_it',
  '1486650_it',
  '1498690_it',
  '1526466_it',
  '15455_it',
  '1565152_it',
  '1706077_it',
  '1769015_it',
  '1777967_it',
  '1803820_it',
  '1833659_it',
  '243838_it',
  '333752_it',
  '382962_it',
  '419929_it',
  '530889_it',
  '548625_it',
  '572916_it',
  '597775_it',
  '606486_it',
  '614605_it',
  '627570_it',
  '63875_it',
  '643686_it',
  '646228_it',
  '646639_it',
  '667544_it',
  '681196_it',
  '709578_it',
  '769230_it',
  '772892_it',
  '785051_it'})

In [ ]:
#TODO clean this function
def crf_dataset():
    with open(OUTPUT_PATH, "a") as f: #TODO this is append now
        for item in ds:
            doc_id = item["document_id"]
            clinical_note = item["clinical_note"]
            if doc_id in processed_ids:
                print(f"Skipped {doc_id} because already processed")
                continue
            success = False
            
            print(f"Processing document: {doc_id}")

            for attempt in range(MAX_RETRIES):
                if "gemma" in MODEL_ID:
                    sleep(20)

                try:
                    # 1. Invoke model                    
                    output = invoke_model(
                        client,
                        MODEL_ID,
                        final_message=f"Now process the following clinical note:\n{clinical_note}",
                        messages=base_messages.copy(),
                        attempt=attempt,
                        system_instruction=sys_instruction,
                        pydantic_model=CRFModel
                    )

                    if isinstance(output, BaseModel):
                        result = output
                    else:
                        # 2. Parse JSON string
                        data_dict = json.loads(output)
                        # 3. Validate with Pydantic
                        result = CRFModel.model_validate(data_dict)
                    
                    # 4. Save
                    save_prediction(f, doc_id, result)
                    success = True
                    break # Exit retry loop on success
                except json.JSONDecodeError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] JSON Error for {doc_id}: {e}")
                    print(f"FAILED OUTPUT:\n{output}\n")
                    # Retry
                
                except pydantic.ValidationError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Validation Error for {doc_id}:")
                    print(e)
                    print(f"FAILED OUTPUT:\n{output}\n")
                    # Retry
                
                except Exception as e:
                    if "The model is overloaded" in str(e) or "This model is currently experiencing high demand" in str(e):
                        print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Service Unavailable for {doc_id}: {e}")
                        attempt-=1
                        sleep(15)
                        continue
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] System/API Error for {doc_id}: {e}")
                    return # TODO currently stopping in case the tokens for that day are over

            if not success:
                print(f"Skipping document {doc_id} after {MAX_RETRIES} failed attempts.")
                
                # Fallback: Create fake instance with all "unknown"
                fake_instance = CRFModel()
                save_prediction(f, doc_id, fake_instance)
        print("Done")


crf_dataset()

Skipped 1014081_it because already processed
Skipped 1117595_it because already processed
Skipped 114284_it because already processed
Skipped 1147182_it because already processed
Skipped 1166267_it because already processed
Skipped 1231372_it because already processed
Skipped 1236585_it because already processed
Skipped 1258026_it because already processed
Skipped 1307609_it because already processed
Skipped 1323348_it because already processed
Skipped 1323973_it because already processed
Skipped 1350262_it because already processed
Skipped 1486650_it because already processed
Skipped 1498690_it because already processed
Skipped 1526466_it because already processed
Skipped 15455_it because already processed
Skipped 1565152_it because already processed
Skipped 1706077_it because already processed
Skipped 1769015_it because already processed
Skipped 1777967_it because already processed
Skipped 1803820_it because already processed
Skipped 1833659_it because already processed
Skipped 24383

In [ ]:
os.chdir("CRF-filling-CL4Health2026")
!python3 scoring.py --submission_path your_submission_data/{OUTPUT_FILE} --language {LANGUAGE}


=== Scoring program starting ===
Loading reference from development_data/dev_gt.jsonl
Loading submission from your_submission_data/gemini-3-flash-preview_new-prompt_it.jsonl
TP: 141, FP: 195, FN: 26
Final macro-F1 = 0.6036701657341567
Scores written to your_sumbmission_scores/scores.json
=== Scoring program finished successfully ===

